# Enrichment Notebook (Scenario 3)
Enriches order data with customer segments. Sets return value for downstream report task.

In [ ]:
import sys
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import session as snowpark_session

try:
    session = get_active_session()
except Exception:
    session = next(iter(snowpark_session._active_sessions))

TARGET_DB = 'AGENTHIVE_LINEAGE'
TARGET_SCHEMA = 'CODE_BUNDLE_DEMO'

user_args = [a for a in sys.argv[1:] if not a.startswith('-') and not a.startswith('/')]
if len(user_args) > 0:
    TARGET_DB = user_args[0]

print(f"Enrichment notebook started. TARGET_DB={TARGET_DB}")

In [ ]:
# Create enriched orders with customer segments
session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.ENRICHED_ORDERS AS
SELECT 
  o.*,
  CASE 
    WHEN o.AMOUNT > 300 THEN 'Premium'
    WHEN o.AMOUNT > 150 THEN 'Standard'
    ELSE 'Budget'
  END AS CUSTOMER_SEGMENT
FROM {TARGET_DB}.{TARGET_SCHEMA}.RAW_ORDERS o
""").collect()

OUTPUT_TABLE = f'{TARGET_DB}.{TARGET_SCHEMA}.ENRICHED_ORDERS'
ROW_COUNT = session.sql(f'SELECT COUNT(*) AS C FROM {OUTPUT_TABLE}').collect()[0]['C']
print(f"Enriched orders: {OUTPUT_TABLE} ({ROW_COUNT} rows)")

In [ ]:
# Set return value for downstream task
return_payload = f"{OUTPUT_TABLE}|{ROW_COUNT}|ENRICHMENT|SUCCESS"
try:
    session.sql(f"SELECT SYSTEM$SET_RETURN_VALUE('{return_payload}')").collect()
    print(f"Return value set: {return_payload}")
except Exception as e:
    if 'must be called from within a task' in str(e):
        print(f"Interactive mode - skipping (payload: {return_payload})")
    else:
        raise